[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](http://colab.research.google.com/github/AcademiXBase/deep-learning-from-scratch/blob/dev/my_notebooks/ch02.ipynb)

In [ ]:
import os
import sys

# 実行環境の確認
if "google.colab" in sys.modules:
    print("✓ Running in Google Colab")
else:
    print("✗ Running locally (not Colab)")

print(f"Python version: {sys.version}")
print(f"Executable: {sys.executable}")
print(f"Working Directory: {os.getcwd()}")

In [ ]:
# Colab で実行している場合、リポジトリをクローンする (2025/12/13 時点での最新ブランチは dev)
#!git clone -b dev https://github.com/AcademiXBase/deep-learning-from-scratch.git
#%cd deep-learning-from-scratch/my_notebooks

In [ ]:
# 自作モジュールのインポート
from modules.ch02_helpers import Gate, CompositeGate, DecisionBoundaryPlotter

In [ ]:
# 作図機能をインスタンス化
plotter = DecisionBoundaryPlotter(
    xlim=(-0.5, 1.5), # (x_min, x_max)
    ylim=(-0.5, 1.5), # (y_min, y_max)
    figsize=(5, 5)  # 5 inch × 5 inch
)

# ２章 パーセプトロン

パーセプトロン (perceptron)：  

アメリカの研究者であるフランク・ローゼンブラットによって 1957 年に考案されたアルゴリズム [[1][1]] で、ニューラルネットワークの基礎となりました。

[1]:https://psycnet.apa.org/record/1959-09865-001

## パーセプトロンとは

複数の信号を入力として受け取り、ひとつの信号を出力するアルゴリズムです。出力される信号は、情報を次に「流す (1) / 流さない (0)」のどちらかの値を取ります。  

ニューロンでは、入力信号にそれぞれ固有の重み ($w_1 x_1, w_2 x_2$) が掛け合わされ、その総和がある閾値 ($\theta$) を超えた場合にのみ信号が伝達されます。この信号が伝達されるという動作が 1 を出力するということに対応し、これを「ニューロンが発火する」と表現することもあります。  

パーセプトロンの動作は式 (2.1) で表されます。  

$$
y =
\begin{cases}
0 & (w_1 x_1 + w_2 x_2 \le \theta) \\
1 & (w_1 x_1 + w_2 x_2 > \theta)
\end{cases}
\tag{2.1}
$$


重みパラメータ ($w_1$ や $w_2$) の大きさは各信号の重要性をコントロールすることに対応し、重みが大きいほど対応する信号が重要視されていることを意味します。  

式 (2.1) は素直でわかりやすいのですが、これ以降のことを考え、すべてのパラメータを左辺へと移行します。ここで、式 (2.1) と式 (2.2) の関係は $\theta = -b$ と表せます。  

$$
y=
\begin{cases}
0 & (b + w_1 x_1 + w_2 x_2 \le 0)\\
1 & (b + w_1 x_1 + w_2 x_2 > 0)
\end{cases}
\tag{2.2}
$$

式 (2.1) と式 (2.2) は記号の表記を変えただけで、まったく同じこと表しています。ここで、$b$ を**バイアス**と呼び、$w_1$ や $w_2$ を**重み**と呼びます。式 (2.2) のような実装の場合、パーセプトロンでは入力信号に重みが乗算された値とバイアスの和が計算され、その値が 0 を上回っていれば 1 を出力し、そうでなければ 0 を出力します。  

Numpy を使った AND ゲートの実装を次に示します。  

```python
def AND(x1, x2):
    x = np.array([x1, x2])
    w = np.array([0.5, 0.5])
    b = -0.7
    tmp = np.sum(w*x) + b
    if tmp <= 0:
        return 0
    else:
        return 1
```

ここで、重みパラメータの $w_1$ や $w_2$ とバイアスの $b$ は別の働きをします。具体的には、重みは入力信号の重要度をコントロールするパラメータとして機能しますが、バイアスは発火のしやすさを調整するパラメータとして機能します。  

また、NAND ゲートと OR も重みとバイアスを変更するだけで簡単に実装できます。  

## 単純な論理ゲート

### AND ゲート

この章では、パーセプトロンを使った簡単な分類問題の例として 2 入力 1 出力をもつ論理ゲートの実装に挑戦します。まず初めに、AND ゲートについて考えてみます。論理ゲートの入力信号と出力信号の対応表を「真理値表」とよび、AND ゲートの場合には次のようになります。

| $x_1$ | $x_2$ | $y$ |
|---:|---:|--:|
| 0  | 0  | 0 |
| 1  | 0  | 0 |
| 0  | 1  | 0 |
| 1  | 1  | 1 |

In [ ]:
Dummy = Gate([1.0, 1.0], -10.0, name="True Table")  # ダミーゲート
fig_and, ax_and = plotter.plot(Dummy)  # 真理値表の各要素を座標上に表現

AND ゲートの真理値表を満たすような $w_1, w_2, b$ の選び方は無限にありますが、例えば $(w_1, w_2, b) = (0.5, 0.5, -0.7)$ などが AND ゲートの条件を満たします。

In [ ]:
# AND ゲートの定義
w = [0.5, 0.5]
b = -0.7
AND = Gate(w, b, name="AND")

In [ ]:
# 出力の確認
for xs in [(0, 0), (1, 0), (0, 1), (1, 1)]:
    y = AND(xs[0], xs[1])
    print(str(xs) + " -> " + str(y))

In [ ]:
# AND ゲートの決定境界をプロット
fig_and, ax_and = plotter.plot(AND)

これを数式で表すと次のようになる。  

$$
y=
\begin{cases}
0 & (-0.7 + 0.5 \, x_1 + 0.5 \, x_2 \le 0)\\
1 & (-0.7 + 0.5 \, x_1 + 0.5 \, x_2 > 0)
\end{cases}
$$

### 2.2.2 NAND ゲートと OR ゲート

続いて、NAND ゲートを考えます。NAND とは、Not AND を意味し、その動作は AND ゲートの出力を逆にしたものになります。真理値表で表すと次のようになります。

| $x_1$ | $x_2$ | $y$ |
|---:|---:|--:|
| 0  | 0  | 1 |
| 1  | 0  | 1 |
| 0  | 1  | 1 |
| 1  | 1  | 0 |

In [ ]:
Dummy = Gate([1.0, 1.0], -10.0, name="True Table")  # ダミーゲート
fig_and, ax_and = plotter.plot(Dummy)  # 真理値表の各要素を座標上に表現

NAND ゲートを表現するパラメータの組み合わせも無限に考えられますが、AND ゲートを実現するパラメータの符号をすべて反転するだけで実現することも可能です。つまり、$(w_1, w_2, b) = (-0.5, -0.5, 0.7)$ などが NAND ゲートの条件を満たします。

In [ ]:
w = [-0.5, -0.5]
b = 0.7
NAND = Gate(w, b, name="NAND")

In [ ]:
for xs in [(0, 0), (1, 0), (0, 1), (1, 1)]:
    y = NAND(xs[0], xs[1])
    print(str(xs) + " -> " + str(y))

In [ ]:
fig_or, ax_or = plotter.plot(NAND)

同様に、これを数式で表現すると次のようになる。

$$
y=
\begin{cases}
0 & (0.7 - 0.5 \, x_1 - 0.5 \, x_2 \le 0)\\
1 & (0.7 - 0.5 \, x_1 - 0.5 \, x_2 > 0)
\end{cases}
$$

同様にして、OR ゲートについても考えます。OR ゲートは入力信号の少なくともひとつが 1 であれば出力が 1 になる論理回路です。真理値表は次のようになります。

| $x_1$ | $x_2$ | $y$ |
|---:|---:|--:|
| 0  | 0  | 0 |
| 1  | 0  | 1 |
| 0  | 1  | 1 |
| 1  | 1  | 1 |

In [ ]:
Dummy = Gate([1.0, 1.0], -10.0, name="True Table")  # ダミーゲート
fig_and, ax_and = plotter.plot(Dummy)  # 真理値表の各要素を座標上に表現

OR ゲートを実現するためのパラメータも、これまでと同様に無限の組み合わせが考えられますが、例えば $(w_1, w_2, b) = (0.5, 0.5, -0.2)$ などで実現可能です。

In [ ]:
OR   = Gate([0.5, 0.5], -0.2, name="OR")

In [ ]:
for xs in [(0, 0), (1, 0), (0, 1), (1, 1)]:
    y = OR(xs[0], xs[1])
    print(str(xs) + " -> " + str(y))

In [ ]:
fig_or, ax_or = plotter.plot(OR)

OR ゲートは次の式で表されます。  

$$
y=
\begin{cases}
0 & (- 0.2 + 0.5 \, x_1 + 0.5 \, x_2 \le 0)\\
1 & (- 0.2 + 0.5 \, x_1 + 0.5 \, x_2 > 0)
\end{cases}
$$

以上のように、パーセプトロンをつかった単純な分類問題として論理ゲートの動作を再現できることができました。ここで重要な点は、AND、NAND、OR ゲートのすべてでパーセプトロンの構造が同じだということです。実際、これらのゲートで異なるものはパラメータ (重みと閾値) の値だけでした。  

ここでは、これらのパーセプトロンのパラメータを手動で決定しましたが、機械学習の問題では、このパラメータの値を決める作業をコンピューターに自動で行わせます。このような仕組みを**学習**とよび、コンピューターが与えられた学習データから適切なパラメータを自動的に求める方法をこの後の章で学びます。  

## パーセプトロンの限界

次に、XOR ゲートと呼ばれる論理回路について考えてみます。  

### XOR ゲート

XOR ゲートは排他的論理和とも呼ばれる論理回路で、次の真理値表に示すように、$x_1$ と $x_2$ のどちらかが 1 のときだけ出力が 1 になります。

| $x_1$ | $x_2$ | $y$ |
|---:|---:|--:|
| 0  | 0  | 0 |
| 1  | 0  | 1 |
| 0  | 1  | 1 |
| 1  | 1  | 0 |

In [ ]:
Dummy = Gate([1.0, 1.0], -10.0, name="True Table")  # ダミーゲート
fig_and, ax_and = plotter.plot(Dummy)  # 真理値表の各要素を座標上に表現

$(x_1, x_2) = (0, 1)$ および $(1, 0)$ を含む領域のみを一本の直線で区切ることは出来ないため、これまで見てきたパーセプトロンでは、この XOR ゲートを実現することは出来ません。

## 多層パーセプトロン

残念ながら、1 つのパーセプトロンだけでは XOR ゲートを実現することができませんでした。しかし、この課題は複数のパーセプトロンを組み合わせることによって実現することが可能です。  

### 既存ゲートの組み合わせ

XOR ゲートは NAND、OR、AND ゲートを次のように配線することで実現可能です。  

各ゲートの出力をもとに真理値表を埋めてみます。

| $x_1$ | $x_2$ | $s_1$ | $s_2$ | $y$ |
|---:|---:|---:|---:|--:|
| 0  | 0  | 1  | 0  | 0 |
| 1  | 0  | 1  | 1  | 1 |
| 0  | 1  | 1  | 1  | 1 |
| 1  | 1  | 0  | 1  | 0 |

### XOR ゲートの実装

Python では次のように簡単に実装することができます。

```python
def XOR(x1, x2):
    s1 = NAND(x1, x2)
    s2 = OR(x1, x2)
    y = AND(s1, s2)
    return y
```

In [ ]:
NAND = Gate(w=[-0.5, -0.5], b= 0.7, name="NAND")
OR   = Gate(w=[ 0.5,  0.5], b=-0.2, name="OR")
AND  = Gate(w=[ 0.5,  0.5], b=-0.7, name="AND")

# 合成ゲート
# XOR(x1, x2) = AND( NAND(x1, x2), OR(x1, x2) )
XOR = CompositeGate(NAND, OR, AND, name="XOR")

In [ ]:
for xs in [(0, 0), (1, 0), (0, 1), (1, 1)]:
    y = XOR(xs[0], xs[1])
    print(str(xs) + " -> " + str(y))

In [ ]:
fig_xor,  ax_xor  = plotter.plot(XOR)

XOR ゲートは次の式で表されます。

$$
y=
\begin{cases}
0 & \left( \biggr\lbrace
\begin{array}{l}
0.7 - 0.5x_1 - 0.5x_2 \le 0\\
-0.2 + 0.5x_1 + 0.5x_2 \le 0
\end{array}
\right)\\[6pt]
1 & (\textit{otherwise})
\end{cases}
$$

## まとめ

1. パーセプトロンは入力に対して出力を決めるアルゴリズムで、パラメータとして**重み**と**バイアス**を持つ。  

2. 単層パーセプトロンでは AND/OR など線形分離可能な論理回路は表現できるが、XOR は表現できない。  

3. 二層以上の多層パーセプトロンにすると XOR など非線形な領域も表現でき、(理論上は) 計算機を表現できる。  

## Appendix

In [ ]:
# 合成ゲート
# XNOR(x1, x2) = NOT( XOR(x1, x2) ) = NAND( XOR(x1, x2), XOR(x1, x2) )
XNOR = CompositeGate(XOR, XOR, NAND, name="XNOR")

In [ ]:
for xs in [(0, 0), (1, 0), (0, 1), (1, 1)]:
    y = XNOR(xs[0], xs[1])
    print(str(xs) + " -> " + str(y))

In [ ]:
fig_xnor, ax_xnor = plotter.plot(XNOR)